# ChordSense Pro — Model v6 (Boundary-Aware, Bass-First, Full-Song Harmonic Architecture)
GPU: L4 24GB / T4 16GB | **8 Epochs / Phase** | **Zero-Overfitting Guardrails** (LoRA `r=16` layers `8..21`, `0%` synthetic in Phase 2, `DropPath=0.15`, `LabelSmoothing=0.08`, `CQT SpecAugment`, `EMA=0.998`)

- Tự động build **Dataset v4** (`/content/data_v4`) ngay từ `v3` trên Drive (cứu ~2,107 hợp âm họ `6` / `m6` / `6/9` + `add4` / `madd4` / `9sus4`, xoá sạch 7 hợp âm lỗi lý thuyết).
- Tự động warm-start trọng số MERT từ `v5_1_fullsong_p3/best.pt` trên Drive.
- **Luôn chạy cell Setup đầu tiên** sau mỗi lần restart runtime.

## 0. Setup — chạy đầu tiên sau mỗi restart

In [ ]:
import os, shutil, yaml
from pathlib import Path

# ── Mount Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

# ── Config ───────────────────────────────────────────────────
DRIVE_PROJECT  = "/content/drive/MyDrive/ChordSensePro"
REPO_DIR       = "/content/ChordSensePro"
DATA_V3_LOCAL  = "/content/data_v3"
DATA_V4_LOCAL  = "/content/data_v4"
PIANO_PACK_DIR = "/content/v4_piano_pack"
CACHE_P3_SSD   = "/content/v6_emb_cache"  # Local NVMe SSD cho Phase 3 (lưu MERT emb + 96-bin CQT)

P1_CFG_STEM    = "v6_20s_p1"
P2_CFG_STEM    = "v6_20s_p2"
P3_CFG_STEM    = "v6_fullsong_p3"

P1_RUN_NAME    = "v6_20s_p1"
P2_RUN_NAME    = "v6_20s_p2"
P3_RUN_NAME    = "v6_fullsong_p3"

# ── Clone / pull repo ─────────────────────────────────────────
if not os.path.exists(REPO_DIR):
    !git clone --branch qkhenh https://github.com/qkhenh/ChordSensePro.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin qkhenh

os.chdir(os.path.join(REPO_DIR, "AI"))
print("cwd:", os.getcwd())

# ── Install deps ──────────────────────────────────────────────
!pip install -q -r requirements.txt

def _make_colab_cfg(cfg_stem: str, run_name: str, extra_train: dict | None = None) -> tuple[str, dict]:
    cfg = yaml.safe_load(Path(f"configs/{cfg_stem}.yaml").read_text())
    cfg["data_dir"]       = DATA_V4_LOCAL
    cfg["checkpoint_dir"] = os.path.join(DRIVE_PROJECT, "AI", "results", run_name)
    cfg["results_csv"]    = os.path.join(DRIVE_PROJECT, "AI", "results", run_name, "results.csv")
    if extra_train:
        cfg["train"].update(extra_train)
    os.makedirs(cfg["checkpoint_dir"], exist_ok=True)
    out_cfg = f"configs/colab_{run_name}.yaml"
    Path(out_cfg).write_text(yaml.dump(cfg, default_flow_style=False))
    return out_cfg, cfg

colab_p1_cfg, cfg_p1 = _make_colab_cfg(P1_CFG_STEM, P1_RUN_NAME)
colab_p2_cfg, cfg_p2 = _make_colab_cfg(P2_CFG_STEM, P2_RUN_NAME)
colab_p3_cfg, cfg_p3 = _make_colab_cfg(P3_CFG_STEM, P3_RUN_NAME, extra_train={"cache_dir": CACHE_P3_SSD})

P1_BEST       = os.path.join(cfg_p1["checkpoint_dir"], "best.pt")
P2_BEST       = os.path.join(cfg_p2["checkpoint_dir"], "best.pt")
P3_BEST       = os.path.join(cfg_p3["checkpoint_dir"], "best.pt")
OUT_ONNX      = os.path.join(DRIVE_PROJECT, "AI", "results", f"model_{P3_RUN_NAME}.onnx")
OUT_EVAL_JSON = os.path.join(DRIVE_PROJECT, "AI", "results", P3_RUN_NAME, "test_metrics.json")

WARM_CANDIDATES = [
    os.path.join(DRIVE_PROJECT, "AI", "results", "v5_1_fullsong_p3", "best.pt"),
    os.path.join(DRIVE_PROJECT, "AI", "results", "v5_1_20s_p2", "best.pt"),
    os.path.join(DRIVE_PROJECT, "AI", "results", "v5_fullsong_p3", "best.pt"),
]
WARM_CKPT = next((p for p in WARM_CANDIDATES if os.path.exists(p)), "")

print("Warm-start ckpt:", WARM_CKPT or "None (train fresh)")
print("P1 epochs:", cfg_p1["train"]["epochs"], "| P2 epochs:", cfg_p2["train"]["epochs"], "| P3 epochs:", cfg_p3["train"]["epochs"])
print("P1 best.pt exists:", os.path.exists(P1_BEST))
print("P2 best.pt exists:", os.path.exists(P2_BEST))
print("P3 best.pt exists:", os.path.exists(P3_BEST))


In [ ]:
!nvidia-smi

## 1. Sync & Build Dataset v4 (`/content/data_v4`)
Tự động copy `v3` từ Drive (nếu chưa có `v4` sẵn trên Drive), chạy `export_dataset_v4.py` để cứu họ hợp âm `6` / `m6` / `6/9`, `add4` / `madd4`, `9sus4`, và ghép thêm `v4_piano_pack` (POP909 + 9 hợp âm cực hiếm) nếu có trên Drive.

In [ ]:
def _audio_ok(d):
    p = os.path.join(d, "train", "audio")
    return os.path.isdir(p) and len(os.listdir(p)) > 0

DATA_V4_DRIVE = os.path.join(DRIVE_PROJECT, "DE", "data", "export", "v4")
DATA_V3_DRIVE = os.path.join(DRIVE_PROJECT, "DE", "data", "export", "v3")
PACK_DRIVE    = os.path.join(DRIVE_PROJECT, "DE", "data", "export", "v4_piano_pack")

if _audio_ok(DATA_V4_LOCAL) and os.path.exists(os.path.join(DATA_V4_LOCAL, "train", "chords.parquet")):
    print("Dataset v4 already ready at", DATA_V4_LOCAL)
elif _audio_ok(DATA_V4_DRIVE):
    print("Copying Dataset v4 directly from Drive...")
    !rm -rf {DATA_V4_LOCAL}
    !cp -r {DATA_V4_DRIVE} {DATA_V4_LOCAL}
else:
    if not _audio_ok(DATA_V3_LOCAL):
        print("Copying base v3 from Drive ->", DATA_V3_LOCAL)
        !rm -rf {DATA_V3_LOCAL}
        !cp -r {DATA_V3_DRIVE} {DATA_V3_LOCAL}
    pack_arg = ""
    if _audio_ok(PACK_DRIVE):
        print("Copying optional POP909 + Rare pack from Drive...")
        !cp -r {PACK_DRIVE} {PIANO_PACK_DIR}
        pack_arg = f"--piano-pack-dir {PIANO_PACK_DIR}"
    else:
        # Tạo nhanh controlled rare-chord pack (~72 bài, <=4% tổng data, 0% ở Phase 2)
        !python ../DE/scripts/build_v4_piano_and_rare_pack.py --skip-pop909 --out {PIANO_PACK_DIR} --rare-songs-per-type 6
        pack_arg = f"--piano-pack-dir {PIANO_PACK_DIR}"
    !python ../DE/scripts/export_dataset_v4.py --v3-dir {DATA_V3_LOCAL} --out {DATA_V4_LOCAL} {pack_arg}

import pandas as pd
df = pd.read_parquet(os.path.join(DATA_V4_LOCAL, "train", "chords.parquet"))
n_audio = len(os.listdir(os.path.join(DATA_V4_LOCAL, "train", "audio")))
sixth_n = int(((df["skel14_id"] == 1) | (df["skel14_id"] == 6)).sum())
slash_pct = 100.0 * (df["bass_id"] != 12).mean()
print(f"v4 train rows: {len(df):,} | 6th-family rescued: {sixth_n:,} | slash: {slash_pct:.2f}% | audio files: {n_audio}")


---
## Phase 1 (8 Epochs) — Warmup v6 Architecture (Frozen MERT + 96-bin CQT + DynamicBoundaryCRF)

In [ ]:
if WARM_CKPT:
    !python -m src.train --config {colab_p1_cfg} --warm_ckpt {WARM_CKPT}
else:
    !python -m src.train --config {colab_p1_cfg}

In [ ]:
# Resume Phase 1 nếu bị disconnect
P1_LAST = os.path.join(cfg_p1["checkpoint_dir"], "last.pt")
if os.path.exists(P1_LAST) and not os.path.exists(P1_BEST):
    !python -m src.train --config {colab_p1_cfg} --resume {P1_LAST}
else:
    print("P1 best.pt:", os.path.exists(P1_BEST), "| last.pt:", os.path.exists(P1_LAST))

---
## Phase 2 (8 Epochs) — Controlled LoRA (`r=16`, MERT layers `8..21` ONLY, `0%` Synthetic Rare Data)

In [ ]:
print("P1 best.pt:", os.path.exists(P1_BEST))
assert os.path.exists(P1_BEST), "Phase 1 chưa xong!"
!python -m src.train --config {colab_p2_cfg} --phase1_ckpt {P1_BEST}

In [ ]:
# Resume Phase 2 nếu bị disconnect
P2_LAST = os.path.join(cfg_p2["checkpoint_dir"], "last.pt")
if os.path.exists(P2_LAST) and not os.path.exists(P2_BEST):
    !python -m src.train --config {colab_p2_cfg} --resume {P2_LAST}
else:
    print("P2 best.pt:", os.path.exists(P2_BEST), "| last.pt:", os.path.exists(P2_LAST))

---
## Phase 3 (8 Epochs) — 60s Full-Song Cross-Section & Dynamic CRF Polish (Cached Emb + 96-bin CQT)

In [ ]:
print("P2 best.pt:", os.path.exists(P2_BEST))
assert os.path.exists(P2_BEST), "Phase 2 chưa xong!"

cfg3 = yaml.safe_load(Path(colab_p3_cfg).read_text())
cfg3["train"]["batch_size"] = 4
cfg3["train"]["grad_accum_steps"] = 4
Path(colab_p3_cfg).write_text(yaml.dump(cfg3, default_flow_style=False))

!python -m src.train --config {colab_p3_cfg} --phase2_ckpt {P2_BEST}

---
## Export ONNX 2-Stage (v6 Encoder + Full-Song Context Decoder) & Benchmark
> Tải cả 3 file trong `MyDrive/ChordSensePro/AI/results/` về máy local (`AI/results/`):
> 1. `model_v6_fullsong_p3.onnx`
> 2. `model_v6_fullsong_p3_decoder.onnx`
> 3. `model_v6_fullsong_p3.meta.json`

In [ ]:
FINAL_CKPT = P3_BEST if os.path.exists(P3_BEST) else P2_BEST
FINAL_CFG  = colab_p3_cfg if os.path.exists(P3_BEST) else colab_p2_cfg
print("Using checkpoint:", FINAL_CKPT)
!python -m src.export_onnx --config {FINAL_CFG} --checkpoint {FINAL_CKPT} --output {OUT_ONNX} --benchmark
print("ONNX files saved next to:", OUT_ONNX)

---
## Full Benchmark trên Tập `test` (647 bài — Composite Score + Passing Chord F1 + 6th Chord F1 + Slash F1)

In [ ]:
FINAL_CKPT = P3_BEST if os.path.exists(P3_BEST) else P2_BEST
!python -m src.evaluate --config {colab_p2_cfg} --checkpoint {FINAL_CKPT} --split test --output {OUT_EVAL_JSON}